In [ ]:
import torch
import torch.nn as nn

N_POINTS = 100

class ContourModel2_TCN(nn.Module):

    def __init__(self):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv1d(8, 64, 5, padding=2),
            nn.BatchNorm1d(64),
            nn.GELU()
        )

        self.block2 = nn.Sequential(
            nn.Conv1d(64, 64, 5, padding=4, dilation=2),
            nn.BatchNorm1d(64),
            nn.GELU()
        )

        self.block3 = nn.Sequential(
            nn.Conv1d(64, 128, 5, padding=8, dilation=4),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.1)
        )

        self.block4 = nn.Sequential(
            nn.Conv1d(128, 128, 5, padding=16, dilation=8),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.1)
        )

        self.refine1 = nn.Sequential(
            nn.Conv1d(128, 128, 3, padding=1),
            nn.BatchNorm1d(128),
            nn.GELU()
        )

        self.refine2 = nn.Sequential(
            nn.Conv1d(128, 64, 3, padding=1),
            nn.BatchNorm1d(64),
            nn.GELU()
        )

        self.output_layer = nn.Conv1d(
            64,
            2,
            kernel_size=1
        )

    def forward(
        self,
        prev_pts,
        future_pts,
        motion,
        n_enc=None
    ):

        velocity = motion * 0.5

        x = torch.cat(
            [
                prev_pts,
                future_pts,
                motion,
                velocity
            ],
            dim=-1
        )

        x = x.permute(0, 2, 1)

        x = self.block1(x)

        r = x
        x = self.block2(x)
        x = x + r

        x = self.block3(x)

        r = x
        x = self.block4(x)
        x = x + r

        x = self.refine1(x)
        x = self.refine2(x)

        x = self.output_layer(x)

        return x.permute(0, 2, 1)